# Hybrid Search in RAG

## What This Notebook Covers

This notebook builds a complete Hybrid Search RAG (Retrieval Augmented Generation) pipeline from scratch. It is organised into three major sections.

Section 1 builds a TF-IDF sparse keyword search system from scratch so you understand exactly what happens at the mathematical level before using library abstractions.

Section 2 demonstrates dense semantic search using pre-computed embedding vectors and shows how cosine similarity ranks results by meaning rather than by exact word overlap.

Section 3 assembles a full production-style RAG pipeline using a real PDF document, ChromaDB for vector storage, BM25 for keyword retrieval, an EnsembleRetriever that fuses both signals, and a local LLM to generate the final answer.

## Why Hybrid Search Exists

A plain RAG system uses only one retrieval method. Keyword search such as TF-IDF and BM25 is excellent at finding documents that contain the exact words the user typed but completely fails when the same idea is expressed with different vocabulary. Semantic vector search understands meaning and handles paraphrases but sometimes misses documents that contain a very specific rare technical term.

Hybrid search combines both signals using a weighted formula so retrieval gets the precision of exact keyword matching and the intelligence of semantic understanding at the same time.

## Section 1  TF-IDF Sparse Keyword Search

### What TF-IDF Does

TF-IDF stands for Term Frequency Inverse Document Frequency. It converts a piece of text into a sparse numerical vector where most values are zero and non-zero values represent how important each word is in that specific document.

Term Frequency counts how often a word appears inside one particular document. A word appearing five times is considered more important than one appearing once.

Inverse Document Frequency penalises words that appear in almost every document because common words like "the" or "is" carry almost no information. Words that appear in only one or two documents receive a high IDF score because they are discriminative and characteristic.

The final TF-IDF score for any word in any document is TF multiplied by IDF. A word scores high only if it is both frequent in that particular document and rare across the full collection. That combination identifies words that are genuinely characteristic of a document rather than just common filler words.

### Why Cosine Similarity

After converting text to TF-IDF vectors we need a way to measure how similar two vectors are. Cosine similarity measures the angle between two vectors rather than the raw distance between their endpoints. This matters for text because two documents about the same topic might have very different lengths. Raw distance would make them look far apart even though they discuss identical subjects. Cosine similarity ignores vector magnitude so a short paragraph and a long chapter about the same topic still score close to 1.0.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# TfidfVectorizer learns a vocabulary from all documents and converts text into sparse TF-IDF vectors
# cosine_similarity measures the angle-based similarity between two vectors, returning values from 0.0 to 1.0
# numpy provides the array operations used for sorting and indexing the ranked results

### The Sample Knowledge Base

These four sentences act as a mini knowledge base. In a real RAG system this would be thousands of chunks extracted from PDFs or databases. The documents all share overlapping vocabulary around keywords and search which will let us observe exactly how TF-IDF scores and ranks them when given a relevant query.

In [2]:
# Sample documents
documents = [
    "This is a list which containig sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings."
]

# This is the question a user might type
# The goal is to find which of the four documents above is most relevant to this query
query = "keyword-based search" 

### Why Preprocessing Is Necessary

Before applying TF-IDF we must clean the text. Without preprocessing, "Keyword" and "keyword" would be treated as two completely different words because Python string comparison is case-sensitive. Similarly punctuation like hyphens and periods would be included as part of tokens, so "keyword-based" might fail to match "keyword based". Lowercasing and stripping punctuation ensures that the query and all documents live in the same vocabulary space so they can be meaningfully compared.

In [3]:
import re

def preprocess_text(text):
    # Convert every character to lowercase so "Keyword" and "keyword" become the same token
    text = text.lower()

    # Remove all punctuation using a regular expression
    # The pattern [^\w\s] matches any character that is NOT a word character (letter, digit, underscore)
    # and NOT a whitespace character. re.sub replaces every such character with an empty string.
    text = re.sub(r'[^\w\s]', '', text)

    return text

In [4]:
# Apply the same preprocessing function to every document in the knowledge base
# List comprehension runs preprocess_text on each string and collects the cleaned results
preprocess_documents = [preprocess_text(doc) for doc in documents]

In [5]:
preprocess_documents

['this is a list which containig sample documents',
 'keywords are important for keywordbased search',
 'document analysis involves extracting keywords',
 'keywordbased search relies on sparse embeddings']

In [6]:
# Print the cleaned documents so we can visually verify preprocessing worked correctly
print("Preprocessed Documents:")
for doc in preprocess_documents:
    print(doc)

Preprocessed Documents:
this is a list which containig sample documents
keywords are important for keywordbased search
document analysis involves extracting keywords
keywordbased search relies on sparse embeddings


In [7]:
# Show the original query before preprocessing for comparison
print("Preprocessed Query:")
print(query)

Preprocessed Query:
keyword-based search


In [8]:
# Preprocess the query using the exact same function used on the documents
# This is critical. The query and documents must go through identical cleaning
# so they share the same vocabulary space. If the documents were lowercased
# but the query was not, the vectors would be incompatible.
preprocessed_query = preprocess_text(query)

In [9]:
preprocessed_query

'keywordbased search'

### Building the TF-IDF Matrix

TfidfVectorizer performs two operations when you call fit_transform. First it scans all documents and builds a vocabulary of every unique word it encounters. Second it converts each document into a numerical vector using that vocabulary. The result is a matrix with one row per document and one column per unique vocabulary word. Each cell contains the TF-IDF score of that word in that document.

The matrix is stored in sparse format which only records non-zero values to save memory. Most cells are zero because most words do not appear in most documents. Calling toarray converts it to a regular dense numpy array so we can inspect the actual numbers.

In [10]:
# Create the TfidfVectorizer object
# At this point it has learned nothing yet. It is just configured and ready.
vector = TfidfVectorizer()

In [11]:
# fit_transform performs two steps in one call:
# Step 1 fit: scans all preprocessed documents and builds the vocabulary of unique words
# Step 2 transform: converts every document into a TF-IDF vector using that vocabulary
# The result X is a sparse matrix with shape (number_of_documents, vocabulary_size)
X = vector.fit_transform(preprocess_documents)

In [12]:
# Convert the sparse matrix to a dense array so we can see all values including zeros
# Each row represents one document, each column represents one unique vocabulary word
# A larger number means that word is more characteristic of that particular document
X.toarray()

array([[0.        , 0.        , 0.37796447, 0.        , 0.37796447,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.37796447, 0.        , 0.        , 0.37796447, 0.        ,
        0.        , 0.37796447, 0.        , 0.        , 0.37796447,
        0.37796447],
       [0.        , 0.4533864 , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.4533864 , 0.4533864 , 0.        ,
        0.        , 0.35745504, 0.35745504, 0.        , 0.        ,
        0.        , 0.        , 0.35745504, 0.        , 0.        ,
        0.        ],
       [0.46516193, 0.        , 0.        , 0.46516193, 0.        ,
        0.        , 0.46516193, 0.        , 0.        , 0.46516193,
        0.        , 0.        , 0.36673901, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.43671931, 0.        , 0.        , 0.       

In [13]:
# Inspect just the TF-IDF vector for the first document
# Most values are zero because document 0 does not contain most vocabulary words
X.toarray()[0]

array([0.        , 0.        , 0.37796447, 0.        , 0.37796447,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.37796447, 0.        , 0.        , 0.37796447, 0.        ,
       0.        , 0.37796447, 0.        , 0.        , 0.37796447,
       0.37796447])

### Transforming the Query Into the Same Space

We use transform here, not fit_transform. This distinction is essential. fit_transform would create a brand new vocabulary from only the query words which would be completely incompatible with the document matrix. By using transform we force the query to be expressed in the same vocabulary that was already learned from the documents. This ensures the query vector and all document vectors have identical dimensions and can be meaningfully compared.

In [14]:
# Transform the query using the vocabulary already learned from the documents
# The output is a sparse matrix with shape (1, vocabulary_size)
# This query vector now lives in exactly the same mathematical space as the document vectors
query_embedding = vector.transform([preprocessed_query])

In [15]:
# Inspect the query vector to see which vocabulary words it activates and with what scores
query_embedding.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.70710678, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.70710678, 0.        , 0.        ,
        0.        ]])

### Ranking Documents by Similarity

cosine_similarity compares every document vector against the query vector and produces a score between 0.0 and 1.0 for each document. A score of 1.0 means the vectors point in exactly the same direction which indicates maximum relevance. A score of 0.0 means the vectors are perpendicular which indicates no shared vocabulary at all.

After computing scores, np.argsort returns the indices that would sort the similarity array. By default argsort sorts in ascending order so the slice notation reverses it to descending order so the most relevant document appears first. The flatten call converts the 2D result array into a simple 1D list of index positions.

In [16]:
# Compare every document vector against the query vector
# The output has shape (number_of_documents, 1) with one similarity score per document
similarities = cosine_similarity(X, query_embedding)

In [17]:
# Inspect the raw similarity scores before ranking
# The document with the highest score is the most relevant to the query
similarities

array([[0.        ],
       [0.50551777],
       [0.        ],
       [0.48693426]])

In [18]:
# Preview the raw argsort output before reversing
# This gives indices in ascending order meaning the least relevant document comes first
np.argsort(similarities, axis=0)

array([[0],
       [2],
       [3],
       [1]])

In [19]:
# Ranking
# [::-1] reverses the sorted order to get descending so the most relevant index appears first
# .flatten() converts the 2D array of shape (n, 1) into a simple 1D array of index values
ranked_indices = np.argsort(similarities, axis=0)[::-1].flatten()

In [20]:
ranked_indices

array([1, 3, 2, 0])

In [21]:
# Use the ranked indices to retrieve the actual document text in order of relevance
ranked_documents = [documents[i] for i in ranked_indices]

In [22]:
# Output the ranked documents
# The most relevant document should appear at Rank 1
for i, doc in enumerate(ranked_documents):
    print(f"Rank {i+1}: {doc}")

Rank 1: Keywords are important for keyword-based search.
Rank 2: Keyword-based search relies on sparse embeddings.
Rank 3: Document analysis involves extracting keywords.
Rank 4: This is a list which containig sample documents.


In [23]:
# Print the original query again for easy comparison with the ranked results above
query

'keyword-based search'

## Section 2  Dense Semantic Search Using Embeddings

### Why Keyword Search Alone Is Not Sufficient

TF-IDF only works when the user's query contains words that literally appear in the documents. If a user asks about "automobile performance" but the document discusses "car speed", TF-IDF scores zero because there are no matching tokens even though the two phrases mean essentially the same thing.

Dense embedding models solve this by encoding the meaning of text rather than just counting individual words. A model like sentence-transformers reads a piece of text and outputs a fixed-size vector of floating point numbers, typically 384 or 768 dimensions. The model is trained on massive amounts of text in a way that ensures sentences with similar meanings produce vectors that are geometrically close together in this high-dimensional space, even when they use completely different vocabulary.

In this section we use hand-crafted 5-dimensional example vectors purely to illustrate the concept. In a real production system you would call a model such as BAAI/bge-base-en-v1.5 to generate these vectors automatically from raw text.

In [24]:
# In a real project you would generate embeddings like this:
#
#   from sentence_transformers import SentenceTransformer
#   model = SentenceTransformer("BAAI/bge-base-en-v1.5")
#   document_embeddings = model.encode(documents)
#
# Reference: https://huggingface.co/sentence-transformers
#
# Here we manually define 5-dimensional dummy vectors to illustrate the math
# Each row represents one document compressed into 5 numbers that encode its meaning
documents = [
    "This is a list which containig sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings."
]

document_embeddings = np.array([
    [0.634, 0.234, 0.867, 0.042, 0.249],
    [0.123, 0.456, 0.789, 0.321, 0.654],
    [0.987, 0.654, 0.321, 0.123, 0.456]
])

In [25]:
# Sample search query (represented as a dense vector)
# In a real system this would be produced by running the query text through the same embedding model
# Using the same model for both query and documents is essential so they share the same vector space
query_embedding = np.array([[0.789, 0.321, 0.654, 0.987, 0.123]])

In [26]:
# Calculate cosine similarity between query and documents
# The math is identical to the TF-IDF case from Section 1
# The only difference is that these dense vectors encode meaning rather than word counts
similarities = cosine_similarity(document_embeddings, query_embedding)

In [27]:
# Inspect the similarity scores produced by dense search
similarities

array([[0.73558979],
       [0.67357898],
       [0.71517305]])

In [28]:
# Same sorting logic as Section 1: argsort, reverse to descending, flatten to 1D
ranked_indices = np.argsort(similarities, axis=0)[::-1].flatten()
ranked_indices

array([0, 2, 1])

In [29]:
# Output the ranked documents
# In a real system the actual document text would be shown here rather than just index numbers
for i, idx in enumerate(ranked_indices):
    print(f"Rank {i+1}: Document {idx+1}")

Rank 1: Document 1
Rank 2: Document 3
Rank 3: Document 2


## Section 3  Full Hybrid RAG Pipeline With a Real PDF

### What This Section Builds

This section constructs a complete working RAG system. It loads a real PDF document, splits it into chunks small enough for an LLM context window, stores those chunks in a vector database for semantic search, indexes them with BM25 for keyword search, merges both retrievers into a hybrid ensemble, and uses a local LLM to read the retrieved chunks and generate a final answer.

### Why We Need a PDF Loader

Raw PDF files are stored in binary format. Before we can work with the text inside a PDF we need a dedicated loader that can parse the binary format, extract the text from each page, and return it as structured Python objects that LangChain can process downstream.

In [30]:
# pypdf is the underlying library that reads PDF binary format and extracts raw text
# langchain_community uses pypdf internally when you call PyPDFLoader
!pip install pypdf

In [31]:
# langchain_community provides document loaders, retrievers, and third-party integrations
# that extend the core LangChain functionality
!pip install langchain_community

In [32]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4772\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [33]:
# Path to the PDF that will serve as our knowledge base
# Adjust this path to wherever you have stored the file on your machine
doc_path = r"../Data/Retrieval-Augmented-Generation-for-NLP.pdf" 

In [34]:
# Initialize the loader with the file path
# At this point the file has not been read yet, the loader is just configured
loader = PyPDFLoader(doc_path)

In [35]:
# Load the PDF and receive a list of LangChain Document objects
# Each Document represents one page of the PDF and contains two attributes:
#   page_content: the raw text extracted from that page as a string
#   metadata: a dictionary with information like source file path and page number
docs = loader.load()

### Why We Split Documents Into Chunks

LLMs have a fixed context window meaning they can only process a limited number of tokens in one call. A typical research paper might be 20 to 30 pages long and contain far more text than fits in any LLM context window. Even if the window were large enough, feeding an entire document as context forces the LLM to read through massive amounts of irrelevant content to find the answer.

Splitting into small chunks lets us retrieve only the specific pieces relevant to the user's question. This makes retrieval both fast and precise. The chunk_overlap parameter ensures that consecutive chunks share a small amount of text so important information at a boundary is never silently cut in half.

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
# chunk_size=200 means each chunk will contain at most 200 characters
# chunk_overlap=30 means the last 30 characters of one chunk will also appear
# at the very start of the next chunk so sentences at boundaries are preserved
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

In [38]:
# Split every page document into smaller chunk documents
# The output is still a list of LangChain Document objects
# Each chunk inherits the metadata from its parent page including page number and source
chunks = splitter.split_documents(docs)

In [39]:
# Preview the chunks to verify splitting produced reasonable sized pieces
chunks

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1'}, page_content='Retrieval-Augmented Generation for Natural\nLanguage Processing: A Survey\nShangyu Wu1,2, Ying Xiong2†, Yufei Cui3, Haolun Wu3, Can\nChen3, Ye Yuan3, Lianming Huang1, Xue Liu2, Tei-Wei Kuo4,'),
 Document(metadata={'producer': 'pikepdf 8.15.1'

In [40]:
# Check how many total chunks were produced from the full document
len(chunks)

1153

### Generating Dense Embeddings With a Real Model

We use BAAI/bge-base-en-v1.5, a strong open-source embedding model specifically optimised for retrieval tasks. BGE stands for BAAI General Embedding and it is developed by the Beijing Academy of Artificial Intelligence. It produces 768-dimensional vectors that capture semantic meaning very well for English text.

HuggingFaceEmbeddings from langchain_huggingface downloads and runs the model locally on your machine without requiring an API key or internet connection after the initial download. Every chunk of text gets converted into a 768-number vector. Chunks about similar topics produce vectors that are geometrically close together in this 768-dimensional space which is exactly what enables semantic search.

In [41]:
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings

In [42]:
# Load the HuggingFace API token from a .env file in the current directory
# The .env file should contain a line that reads: HF_TOKEN=hf_xxxxxxxxxxxxxxxx
# Storing secrets in a .env file keeps them out of the code so the project is safe to push to GitHub
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

print(HF_TOKEN[:10])

hf_OssOODY


In [43]:
# Windows alternative if the above call does not find the .env file automatically
# On Windows the working directory path sometimes differs from what load_dotenv expects
# Providing the absolute path to the .env file solves this
from dotenv import load_dotenv
import os

load_dotenv(r"F:\Advance RAG\.env")

HF_TOKEN = os.getenv("HF_TOKEN")

In [44]:
# chromadb is the vector database that stores our chunk embeddings
# and performs fast approximate nearest-neighbor search when a query arrives
!pip install chromadb

In [45]:
from langchain_chroma import Chroma

In [46]:
from langchain_huggingface import HuggingFaceEmbeddings

In [47]:
# Initialize the embedding model
# model_name: BAAI/bge-base-en-v1.5 is optimised for retrieval and produces 768-dim vectors
# The model weights download from HuggingFace Hub on the first run (approximately 400 MB)
# After that the model runs entirely locally without any further internet calls
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [48]:
# Build the ChromaDB vector store from our document chunks
# This performs three operations internally:
#   1. Takes the text content of every chunk
#   2. Calls the embedding model to compute a 768-dimensional vector for each chunk
#   3. Stores all vectors and their corresponding text in ChromaDB for fast retrieval
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [49]:
# Wrap the vector store as a LangChain retriever
# search_kwargs={"k": 3} tells the retriever to return the 3 most similar chunks per query
# When a query arrives the retriever converts it to an embedding and finds the 3 nearest vectors
vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 3})

In [50]:
# Inspect the retriever object to confirm it was created correctly
vectorstore_retreiver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000224DD2F4830>, search_kwargs={'k': 3})

### Adding BM25 Keyword Retrieval

BM25 (Best Matching 25) is the modern standard algorithm for keyword-based document retrieval. It powers Elasticsearch, Solr, and most traditional search engines. It improves on raw TF-IDF in two important ways.

Document length normalisation ensures a long document does not unfairly outscore a short document simply because it contains more total words. BM25 adjusts scores relative to the average document length across the collection.

Term frequency saturation means that repeating a keyword 100 times does not produce a score 100 times higher than repeating it once. The score increases diminishingly with repeated occurrences which prevents irrelevant documents from ranking highly just by stuffing keywords.

We create a BM25Retriever from the same chunks stored in ChromaDB. Setting k to 3 means it also returns the top 3 results per query matching the vector store retriever so both contribute an equal number of candidates to the ensemble.

In [51]:
# rank_bm25 is the underlying Python library that implements the BM25 ranking algorithm
# LangChain wraps this library inside BM25Retriever for seamless integration
!pip install rank_bm25

In [52]:
from langchain_community.retrievers import BM25Retriever

In [53]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

In [54]:
# Build the BM25 retriever from the same chunks used for the vector store
# BM25Retriever tokenises all chunk texts and indexes them in memory for fast keyword lookup
keyword_retriever = BM25Retriever.from_documents(chunks)

In [55]:
# Tell BM25 to return the top 3 results per query
# This matches the vector store retriever's k=3 so both contribute equally
keyword_retriever.k = 3

### Combining Both Retrievers Into an EnsembleRetriever

EnsembleRetriever accepts a list of retrievers and a list of corresponding weights. When a query arrives it sends the query to all retrievers in parallel, collects their individual ranked result lists, and merges them using Reciprocal Rank Fusion.

Reciprocal Rank Fusion assigns each document a score based on its rank position in each retriever's result list multiplied by the corresponding weight. Documents that appear near the top of multiple retriever lists receive the highest combined scores because they are relevant from multiple perspectives simultaneously.

The weights here are set to 0.3 for the vector store and 0.7 for BM25. This means keyword matching contributes 70 percent of the final ranking signal. For a technical research paper with precise terminology BM25 is generally more reliable because users tend to search for exact technical terms. Adjust these weights based on your specific domain and query patterns.

In [56]:
# Create the hybrid retriever by combining dense and sparse retrieval
# retrievers: list containing the two retriever objects built above
# weights: how much influence each retriever has in the merged ranking
#   0.3 means the vector store (semantic) contributes 30 percent of the ranking signal
#   0.7 means BM25 (keyword) contributes 70 percent of the ranking signal
#   the weights must sum to 1.0
ensemble_retriever = EnsembleRetriever(
    retrievers=[vectorstore_retreiver, keyword_retriever],
    weights=[0.3, 0.7]
)

## The Hybrid Score Formula

hybrid_score equals (1 minus alpha) multiplied by sparse_score plus alpha multiplied by dense_score

When alpha equals 0 the result is pure BM25 keyword search with no semantic component.
When alpha equals 1 the result is pure dense semantic search with no keyword component.
When alpha equals 0.5 both signals contribute equally to the final ranking.

In LangChain EnsembleRetriever the weights parameter directly maps to this formula.
weights=[0.3, 0.7] corresponds to alpha equals 0.3 meaning the dense vector store contributes 30 percent and BM25 contributes 70 percent.

### Loading the Language Model

We use TinyLlama-1.1B-Chat, a compact but capable instruction-tuned language model with 1.1 billion parameters. It is small enough to run on a standard laptop CPU or a free Colab GPU without needing any quantization tricks, making it ideal for learning and experimentation.

AutoModelForCausalLM is the HuggingFace class for loading decoder-only transformer models such as LLaMA, GPT, and Mistral. These models generate text one token at a time by predicting what the most likely next token is given everything that came before it in the context.

AutoTokenizer loads the corresponding tokenizer that knows how to convert raw text into the integer token IDs the model actually processes, and how to convert the output integer IDs back into readable human text.

In [57]:
# bitsandbytes provides 4-bit and 8-bit quantization support
# If you want to run a larger model like Zephyr-7B on limited GPU memory
# quantization is the key tool that compresses model weights to fit
!pip install bitsandbytes

In [58]:
# accelerate handles device placement and distributes model layers across available hardware
# It also speeds up model loading significantly on single-GPU and CPU-only setups
!pip install accelerate

In [59]:
import torch

from transformers import (
    AutoModelForCausalLM,    # Loads any decoder-only causal language model from HuggingFace Hub
    AutoTokenizer,            # Loads the matching tokenizer paired with any HuggingFace model
    BitsAndBytesConfig,       # Configuration object for 4-bit or 8-bit quantization if needed
    pipeline                  # High-level API that wraps model and tokenizer into one inference object
)

from langchain_huggingface import HuggingFacePipeline  # Wraps an HF pipeline as a LangChain LLM

In [60]:
def load_model(model_name: str):
    # AutoModelForCausalLM.from_pretrained downloads model weights from HuggingFace Hub
    # if they are not already cached locally, then loads them into CPU or GPU memory
    # For TinyLlama this downloads approximately 2 GB of weights on the first run
    model = AutoModelForCausalLM.from_pretrained(
        model_name
    )

    return model

In [61]:
# initializing tokenizer
def initialize_tokenizer(model_name: str):
    # AutoTokenizer.from_pretrained loads the tokenizer saved alongside the model on HuggingFace Hub
    # return_token_type_ids=False: TinyLlama and most modern models do not use token type IDs
    # Token type IDs were a BERT-specific feature for distinguishing two different input sentences
    # Setting this to False avoids a harmless but confusing deprecation warning
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        return_token_type_ids=False
    )

    # bos_token_id=1 manually sets the beginning-of-sequence token identifier
    # This tells the model exactly where the start of the input begins
    # and avoids a warning that appears when this is not explicitly set
    tokenizer.bos_token_id = 1

    return tokenizer

In [62]:
# TinyLlama is a 1.1 billion parameter model fine-tuned for chat and instruction following
# It produces reasonable answers while being fast enough to run on consumer hardware
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 

In [63]:
# Load the tokenizer first before the model
# The model configuration references tokenizer settings like vocabulary size
# so the tokenizer should be ready before model loading begins
tokenizer = initialize_tokenizer(model_name)

### Verifying the Tokenizer Before Loading the Heavy Model

It is useful to test what the tokenizer does to text before spending time loading the full model weights. The tokenizer converts a raw string into a dictionary containing input_ids which is a list of integer token IDs, and attention_mask which is a list of ones marking which positions are real tokens versus padding positions.

In [64]:
text = "What is RAG?"

# tokenizer(text) splits the input string into subword tokens and maps each to an integer ID
# The model never sees raw text, it only ever processes these integer IDs
tokens = tokenizer(text)

print(tokens)

{'input_ids': [1, 1724, 338, 390, 10051, 29973], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [65]:
# Download and load the model weights into memory
# This may take several minutes the first time because it downloads from HuggingFace Hub
# Subsequent runs use the cached weights stored on your local disk
model = load_model(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Building the Text Generation Pipeline

The HuggingFace pipeline function wraps the model and tokenizer into one convenient object that handles the complete inference loop: tokenising the input text, running the transformer forward pass, sampling the next token from the output probability distribution, appending that token to the growing sequence, and repeating until a stop condition is reached.

max_new_tokens controls the maximum number of tokens the model is allowed to generate beyond the input. Setting this to 256 gives enough room for a paragraph-length answer without risking runaway generation.

temperature controls how deterministic or random the sampling is. A value of 0.7 sits between fully deterministic at 0.0 and maximally random at 1.0, producing coherent yet naturally varied outputs.

top_k limits sampling at each step to only the 50 most probable candidate tokens. This prevents the model from ever sampling a highly improbable token that would produce incoherent text while still leaving enough variation for natural language generation.

In [66]:
# Create Hugging Face text generation pipeline
pipe = pipeline(
    task="text-generation",       # Instructs the pipeline to generate new text continuing from the input
    model=model,
    tokenizer=tokenizer,
    use_cache=True,               # Caches key-value attention states to speed up token-by-token generation
    max_new_tokens=256,           # Maximum number of tokens to generate beyond the input prompt
    do_sample=True,               # Use probabilistic sampling rather than always picking the single highest-probability token
    temperature=0.7,              # Controls randomness: lower values produce more focused output, higher values more creative
    top_k=50,                     # At each generation step only sample from the 50 most likely next tokens
    num_return_sequences=1,       # Generate one response per call rather than multiple alternatives
    eos_token_id=tokenizer.eos_token_id,   # Stop generation automatically when the end-of-sequence token is produced
    pad_token_id=tokenizer.eos_token_id,   # Reuse end-of-sequence token as the padding token for batched inputs
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_k', 'temperature', 'use_cache', 'pad_token_id', 'num_return_sequences', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [67]:
# Convert Transformers pipeline to LangChain LLM
# HuggingFacePipeline wraps the raw HF pipeline behind the standard LangChain LLM interface
# This allows the model to be plugged into any LangChain chain without modification
llm = HuggingFacePipeline(
    pipeline=pipe
)

In [68]:
# Test the model standalone before connecting it to any retriever
# This confirms the LLM generates sensible text on its own before we add retrieval complexity
response = llm.invoke(
    "What is Retrieval Augmented Generation?"
)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [69]:
print(response)

What is Retrieval Augmented Generation?How does it work?

Kai: The Retrieval Augmented Generation (RAG) model is a type of language model pre-trained on a large corpus of text, such as Wikipedia or Google Books. The model is trained to generate new text that captures the structure and meaning of the original text, while also following the user's input.

The model takes as input a given text and a set of queries, and generates a new piece of text that follows the style and structure of the original text while still being relevant to the user's queries. RAG models have been shown to be effective in many domains, such as information retrieval, machine translation, and natural language generation.

One example of a specific use case is in information retrieval. RAG models can be pre-trained to generate responses to queries based on the structure and meaning of the original text. This can be particularly useful in scenarios where the user's query may not be well-formed or may require more c

### Building the RAG Chains

RetrievalQA is a LangChain chain that connects a retriever to an LLM. When you call invoke with a question it automatically performs three steps. First it sends the question to the retriever to fetch the most relevant document chunks. Then it formats a prompt that includes all retrieved chunks as context followed by the original question. Finally it passes this complete prompt to the LLM and returns the generated answer.

chain_type equals "stuff" means all retrieved chunks are stuffed directly into a single prompt as context. This is the simplest approach and works reliably when the chunks are small enough to fit within the LLM context window, which they are given our chunk_size of 200 characters.

We build two chains with identical LLM settings but different retrievers so we can directly compare the quality of answers produced by pure semantic retrieval versus hybrid retrieval.

In [70]:
from langchain_classic.chains import RetrievalQA

In [71]:
# Normal RAG chain using only the ChromaDB vector store retriever
# This chain finds relevant chunks purely by semantic similarity using dense embeddings
# It handles paraphrases and meaning well but may miss chunks containing rare exact technical terms
normal_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_retreiver
)

In [72]:
# Hybrid RAG chain using the EnsembleRetriever
# This chain finds relevant chunks via both BM25 keyword matching and dense semantic search
# The combined 30 percent semantic plus 70 percent keyword ranking produces more robust retrieval
# than either method alone, especially for technical queries with precise terminology
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever
)

### Comparing Normal RAG Versus Hybrid RAG

We run the same question through both chains to observe the difference in answers. The invoke method returns a dictionary with two keys. The query key contains the original question and the result key contains the LLM generated answer.

By comparing the two outputs you can observe how hybrid retrieval often surfaces richer and more relevant context for the LLM leading to more complete and accurate answers, particularly for specific technical terms that appear in the source document.

In [74]:
# Run the question through the normal vector-only RAG chain
# The LLM will receive as context only the 3 chunks that are most semantically similar to the query
response1 = normal_chain.invoke("What is RAG?")

In [75]:
# Inspect the full response dictionary to see both the query and the result keys
response1

{'query': 'What is RAG?',
 'result': "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nas a practical guide for researchers and practitioners advancing RAG systems.\n12 Statements and Declarations\n12.1 Authors’ Contribution\n\n• RAG in Industry\n• Frameworks and \nLibraries\n• Deployment \nConsiderations\n• Security and Privacy \nConsiderations for RAG\n• Retrieval Quality\n• RAG Efficiency\n• RAG Reliability\n\nattack surface.\nPrivacy Risks of External Knowledge Bases.RAG systems typically store\neither raw documents, embeddings, or both. A common assumption is that embed-\n\nQuestion: What is RAG?\nHelpful Answer:Reusable Annotation Grammar. RAG (Reusable Annotation Grammar) is a research area in natural language processing (NLP) that focuses on the reuse and exchange of annotations in NLP pipelines.\nQuestion: What is a RAG system?\nHelpful Answer: Reusable Annotati

In [76]:
# Print just the generated answer text for easy reading
print(response1.get("result"))

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

as a practical guide for researchers and practitioners advancing RAG systems.
12 Statements and Declarations
12.1 Authors’ Contribution

• RAG in Industry
• Frameworks and 
Libraries
• Deployment 
Considerations
• Security and Privacy 
Considerations for RAG
• Retrieval Quality
• RAG Efficiency
• RAG Reliability

attack surface.
Privacy Risks of External Knowledge Bases.RAG systems typically store
either raw documents, embeddings, or both. A common assumption is that embed-

Question: What is RAG?
Helpful Answer:Reusable Annotation Grammar. RAG (Reusable Annotation Grammar) is a research area in natural language processing (NLP) that focuses on the reuse and exchange of annotations in NLP pipelines.
Question: What is a RAG system?
Helpful Answer: Reusable Annotation Grammar. A RAG system is a software tool that reuses annota

In [78]:
# Run the same question through the hybrid RAG chain
# The LLM will receive chunks found by both keyword matching AND semantic search
# The retrieved context is often richer and more targeted than vector-only retrieval
response2 = hybrid_chain.invoke("What is Abstractive Question Answering?")

In [79]:
# Inspect the full response dictionary from the hybrid chain
response2

{'query': 'What is Abstractive Question Answering?',
 'result': "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n6.4 Question Answering\nT ask Characteristics.Question Answering (QA) is a fundamental task in NLP\nthat involves building systems capable of automatically answering human questions\n\nissues, and (iii) copy bias that harms abstraction and can even import contradictions\nfrom noisy retrievals.\n6.4 Question Answering\n\n𝑙𝑙𝑞𝑞 𝑙𝑙1, 𝑙𝑙2, … , 𝑙𝑙𝑘𝑘\n𝐿𝐿out\n(a) Query-based fusion\nQuery 𝑞𝑞: What is the tallest \nmountain in the world?\nRetriever\nTop-k chunk texts\n𝑧𝑧1, 𝑧𝑧2, … , 𝑧𝑧𝑘𝑘\nText/Feature concatenation\n\nspecific question answering using llms. Natural Language Processing Journal\n7:100065. https://doi.org/10.1016/J.NLP.2024.100065\n\nmented multi-hop question answering. In: Proceedings of the 63rd Annual Meeting\nof the Association for Computational Lingu

In [80]:
# Print the hybrid chain answer for direct comparison with the normal chain answer above
print(response2.get("result"))

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

6.4 Question Answering
T ask Characteristics.Question Answering (QA) is a fundamental task in NLP
that involves building systems capable of automatically answering human questions

issues, and (iii) copy bias that harms abstraction and can even import contradictions
from noisy retrievals.
6.4 Question Answering

𝑙𝑙𝑞𝑞 𝑙𝑙1, 𝑙𝑙2, … , 𝑙𝑙𝑘𝑘
𝐿𝐿out
(a) Query-based fusion
Query 𝑞𝑞: What is the tallest 
mountain in the world?
Retriever
Top-k chunk texts
𝑧𝑧1, 𝑧𝑧2, … , 𝑧𝑧𝑘𝑘
Text/Feature concatenation

specific question answering using llms. Natural Language Processing Journal
7:100065. https://doi.org/10.1016/J.NLP.2024.100065

mented multi-hop question answering. In: Proceedings of the 63rd Annual Meeting
of the Association for Computational Linguistics (ACL), pp 22362–22375

Yang Z, Qi P, Zhang S, et al (2018) Hotpotqa: A dataset for